<a href="https://colab.research.google.com/github/Sultoniromadhon/data-science-2026/blob/main/Pertemuan3_Sultoni_Romadhon_250401020198.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

* Nama : SULTONI ROMADHON
* NIM : 250401020198
* Kelas : IF403
* Prodi : Informatika PJJ S1

In [3]:
# ============================================
# PIPELINE DATA CLEANING — HOUSING DATASET
# ============================================

import pandas as pd
import numpy as np
from scipy.stats.mstats import winsorize

# untuk API
import requests

In [4]:
# STEP 0 — Load & eksplorasi awal
df = pd.read_csv('/content/sample_data/housing_dirty.csv')

In [5]:
print('Shape awal:', df.shape)
print(df.isnull().sum())

Shape awal: (130, 7)
id               0
luas_m2         18
harga_juta      17
kota             0
kamar           10
tahun_bangun     0
kondisi          0
dtype: int64


In [6]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB


,id,luas_m2,harga_juta,kamar,tahun_bangun
count,130.000000,112.000000,1.130000e+02,120.000000,130.000000
mean,65.500000,267.627679,8.856325e+05,3.433333,2062.638462
std,37.671829,885.664181,9.407144e+06,1.776283,701.684043
min,1.000000,-50.000000,-5.000000e+02,1.000000,1890.000000
25%,33.250000,87.050000,3.450000e+02,2.000000,1991.250000
50%,65.500000,193.800000,6.550000e+02,4.000000,2002.000000
75%,97.750000,280.675000,9.550000e+02,5.000000,2011.750000
max,130.000000,9500.000000,1.000000e+08,6.000000,9999.000000


In [7]:
# STEP 1 — Hapus Duplikat
df.drop_duplicates(inplace=True)
print('Setelah hapus duplikat:', df.shape)

Setelah hapus duplikat: (130, 7)


In [8]:
# STEP 2 — Normalisasi String
df['kota'] = df['kota'].astype(str).str.strip().str.title()
df['kondisi'] = df['kondisi'].astype(str).str.strip().str.lower()

In [9]:
# STEP 3 — Imputasi Missing Values
df['luas_m2'] = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())
df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])

In [10]:
# STEP 4 — Tangani Outlier (IQR Fence)
for col in ['harga_juta', 'luas_m2', 'tahun_bangun']:
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower, upper)

In [11]:
# STEP 5 — Validasi & Ekspor
print('Total missing:', df.isnull().sum().sum())
print('Total duplikat:', df.duplicated().sum())

Total missing: 0
Total duplikat: 0


In [12]:
assert df.isnull().sum().sum() == 0, 'Masih ada missing!'
assert df.duplicated().sum() == 0, 'Masih ada duplikat!'

In [13]:
print('Shape akhir:', df.shape)

Shape akhir: (130, 7)


In [14]:

df.to_csv('/content/sample_data/housing_clean.csv', index=False)
print('Dataset bersih tersimpan!')

Dataset bersih tersimpan!


In [15]:
# Akses API JSONPlaceholder dan simpan respons sebagai DataFrame
response = requests.get("https://jsonplaceholder.typicode.com/users")
data = response.json()

df = pd.DataFrame(data)
df.head()

,id,name,username,email,address,phone,website,company
0,1,Leanne Graham,Bret,Sincere@april.biz,"{'street': 'Kulas Light', 'suite': 'Apt. 556',...",1-770-736-8031 x56442,hildegard.org,"{'name': 'Romaguera-Crona', 'catchPhrase': 'Mu..."
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,"{'street': 'Victor Plains', 'suite': 'Suite 87...",010-692-6593 x09125,anastasia.net,"{'name': 'Deckow-Crist', 'catchPhrase': 'Proac..."
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,"{'street': 'Douglas Extension', 'suite': 'Suit...",1-463-123-4447,ramiro.info,"{'name': 'Romaguera-Jacobson', 'catchPhrase': ..."
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,"{'street': 'Hoeger Mall', 'suite': 'Apt. 692',...",493-170-9623 x156,kale.biz,"{'name': 'Robel-Corkery', 'catchPhrase': 'Mult..."
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,"{'street': 'Skiles Walks', 'suite': 'Suite 351...",(254)954-1289,demarco.info,"{'name': 'Keebler LLC', 'catchPhrase': 'User-c..."


In [16]:
url = "https://api.bmkg.go.id/publik/prakiraan-cuaca?adm4=31.71.01.1001"
data = requests.get(url).json()

cuaca = data["data"][0]["cuaca"]

flat = []
for group in cuaca:
    for item in group:
        flat.append(item)

df_bmkg = pd.DataFrame(flat)
df_bmkg.head()

,datetime,t,tcc,tp,weather,weather_desc,weather_desc_en,wd_deg,wd,wd_to,ws,hu,vs,vs_text,time_index,analysis_date,image,utc_datetime,local_datetime
0,2026-06-09T05:00:00Z,32,26,0.0,1,Cerah,Sunny,18,N,S,8.1,51,13573,> 10 km,28-29,2026-06-08T00:00:00,https://api-apps.bmkg.go.id/storage/icon/cuaca...,2026-06-09 05:00:00,2026-06-09 12:00:00
1,2026-06-09T08:00:00Z,31,12,0.1,1,Cerah,Sunny,7,N,S,12.9,56,13928,> 10 km,31-32,2026-06-08T00:00:00,https://api-apps.bmkg.go.id/storage/icon/cuaca...,2026-06-09 08:00:00,2026-06-09 15:00:00
2,2026-06-09T11:00:00Z,30,18,0.0,1,Cerah,Sunny,23,N,S,7.3,62,12796,> 10 km,34-35,2026-06-08T00:00:00,https://api-apps.bmkg.go.id/storage/icon/cuaca...,2026-06-09 11:00:00,2026-06-09 18:00:00
3,2026-06-09T14:00:00Z,29,97,0.0,3,Berawan,Mostly Cloudy,57,NE,SW,3.0,65,12599,> 10 km,37-38,2026-06-08T00:00:00,https://api-apps.bmkg.go.id/storage/icon/cuaca...,2026-06-09 14:00:00,2026-06-09 21:00:00
4,2026-06-09T17:00:00Z,28,96,0.0,3,Berawan,Mostly Cloudy,106,E,W,2.5,69,10935,> 10 km,40-41,2026-06-08T00:00:00,https://api-apps.bmkg.go.id/storage/icon/cuaca...,2026-06-09 17:00:00,2026-06-10 00:00:00


Kesimpulan

Pada notebook ini dilakukan proses data cleaning pada dataset perumahan serta pengolahan data dari API. Tahapan yang dilakukan meliputi penghapusan duplikat, normalisasi data, imputasi missing value, dan penanganan outlier menggunakan metode IQR. Selain itu, data dari API juga diubah menjadi bentuk DataFrame agar dapat dianalisis.

Dari proses ini, data menjadi lebih bersih, konsisten, dan siap digunakan untuk analisis lebih lanjut. Namun, masih terdapat keterbatasan seperti error handling API yang sederhana dan belum adanya pemisahan variabel DataFrame secara optimal.